# **PyTorch Implementation of the same Model**

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer
from datasets import load_dataset

In [3]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [4]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

sentiment/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/validation-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  167kB            

sentiment/validation-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [6]:
print(dataset["train"][0])
print(dataset["test"][0])
print(dataset["validation"][0])

{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}
{'text': "@user @user what do these '1/2 naked pics' have to do with anything? They're not even like that.", 'label': 1}
{'text': 'Dark Souls 3 April Launch Date Confirmed With New Trailer: Embrace the darkness.', 'label': 1}


In [7]:
from collections import Counter
print(Counter(dataset["train"]['label']))
print(Counter(dataset["test"]['label']))
print(Counter(dataset["validation"]['label']))


Counter({1: 20673, 2: 17849, 0: 7093})
Counter({1: 5937, 0: 3972, 2: 2375})
Counter({1: 869, 2: 819, 0: 312})


In [8]:
print(dataset["train"].features["label"])

ClassLabel(names=['negative', 'neutral', 'positive'])


In [9]:
label_feature = dataset["train"].features["label"]

In [10]:
print(label_feature.int2str(0))
print(label_feature.int2str(1))
print(label_feature.int2str(2))

negative
neutral
positive


In [11]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
vocab = tokenizer.get_vocab()

In [14]:
type(vocab)

dict

In [15]:
vocab = tokenizer.get_vocab().items()

In [16]:
len(vocab)

30522

In [17]:
print(tokenizer.__dir__())

['do_lower_case', 'tokenize_chinese_chars', 'strip_accents', '_vocab', '_tokenizer', '_add_bos_token', '_add_eos_token', '_should_update_post_processor', 'init_inputs', 'init_kwargs', 'name_or_path', '_processor_class', '_pad_token_type_id', 'verbose', '_special_tokens_map', '_extra_special_tokens', 'model_max_length', 'padding_side', 'truncation_side', 'model_input_names', 'clean_up_tokenization_spaces', 'clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output', 'split_special_tokens', '_in_target_context_manager', 'chat_template', 'response_schema', 'response_template', 'deprecation_warnings', 'backend', 'files_loaded', 'vocab_file', 'add_prefix_space', '__module__', '__doc__', 'vocab_files_names', 'model', '__init__', 'convert_to_native_format', 'is_fast', 'can_save_slow_tokenizer', 'save_vocabulary', 'update_post_processor', 'add_eos_token', 'add_bos_token', '_post_init', 'vocab_size', 'get_vocab', 'vocab', 'added_tokens_encoder', 'added_tokens_decoder', '_added_tok

In [18]:
vocab = tokenizer.get_vocab()

In [19]:
print(len(vocab))

30522


In [20]:
list(vocab.items())[-10:-1]

[('ordination', 18129),
 ('donation', 13445),
 ('outcome', 9560),
 ('cousin', 5542),
 ('solitary', 14348),
 ('enable', 9585),
 ('floods', 14295),
 ('daly', 18509),
 ('##ios', 10735)]

In [21]:
print(tokenizer.special_tokens_map)

{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


In [22]:
print(tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_type_id)

101 102 0


In [26]:
#let's test the tokenizer
text = "I love this!"
token_ids = tokenizer.encode(text)

In [27]:
token_ids

[101, 1045, 2293, 2023, 999, 102]

In [28]:
print(token_ids)

[101, 1045, 2293, 2023, 999, 102]


In [29]:
print(tokenizer.tokenize("heartful"))

['heart', '##ful']


In [32]:
tokenizer.encode("unbelievability")

[101, 4895, 8671, 2666, 3567, 8553, 102]

In [34]:

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3, id2label=id2label, label2id=label2id)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
print(model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "vocab_size": 30522
}



In [36]:
sum(p.numel() for p in model.parameters()) == sum(p.numel() for p in model.parameters() if p.requires_grad)

True

In [37]:
import torch

batch = tokenizer(["I love this!", "This is terrible."], return_tensors="pt", padding=True, truncation=True)
with torch.no_grad():
    output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

print(output.logits.shape)
print(output.logits)

torch.Size([2, 3])
tensor([[ 0.1440,  0.0403, -0.0371],
        [ 0.1318,  0.0308, -0.0535]])


In [39]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [40]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})


In [41]:
print(tokenized_dataset["train"][0].keys())

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [42]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["text", "token_type_ids"])
tokenized_dataset.set_format("torch")
print(tokenized_dataset)
# print(tokenized_dataset["train"][0].keys())

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
})


In [43]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [44]:
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

In [45]:
sample_batch = [tokenized_dataset["train"][i] for i in range(4)]
batch = data_collator(sample_batch)

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4])


In [46]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    """
    eval_pred: tuple of (logits, labels)
      logits: shape (num_examples, 3) - raw model outputs, before softmax
      labels: shape (num_examples,)   - true class indices (0/1/2)
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # pick the highest-scoring class per example

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

In [47]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=0.2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

In [48]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [49]:
print(next(model.parameters()).device)

cuda:0


In [50]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated(0) / 1e9, "GBs allocated")

True
Tesla T4
0.268960256 GBs allocated


In [51]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
0,0.714768,0.683006,0.689500,0.690576


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.state.log_history

In [53]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

./distilbert-sentiment/checkpoint-571
0.6905764918465566


In [54]:
!ls ./distilbert-sentiment

checkpoint-571


In [55]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0
)

examples = [
    "I absolutely love this new phone, it's amazing!",
    "This is the worst service I've ever experienced.",
    "The meeting is scheduled for 3pm tomorrow.",
]

results = classifier(examples)
for text, result in zip(examples, results):
    print(f"{text}\n → {result}\n")

I absolutely love this new phone, it's amazing!
 → {'label': 'positive', 'score': 0.9272681474685669}

This is the worst service I've ever experienced.
 → {'label': 'negative', 'score': 0.6952694058418274}

The meeting is scheduled for 3pm tomorrow.
 → {'label': 'neutral', 'score': 0.8042048811912537}

